# Solar Orbiter merged MAG availability (daily, non-overlapping)

This notebook is based on the `SOLO.ipynb` workflow, but forces:
- `SOLO_use_merged_MAG = True`
- 1-day non-overlapping intervals
- 1-minute cadence diagnostics for `beta` and `sigma_c`

It also saves the **usual MHDTurbPy files** (`final.pkl`, `general.pkl`, `sig_c_sig_r.pkl`, and gap files) for each successful day.


In [ ]:
from pathlib import Path
import importlib.util
import os
import json

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "functions").is_dir() and (candidate / "pyspedas").is_dir():
            return candidate
    raise RuntimeError("Could not locate MHDTurbPy root (missing functions/ and pyspedas/).")


root_dir = _find_repo_root(Path.cwd())
path_setup_file = root_dir / "functions" / "path_setup.py"
spec = importlib.util.spec_from_file_location("mhdturbpy_path_setup", path_setup_file)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load path setup from {path_setup_file}")
path_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(path_setup)

root_dir = path_setup.ensure_project_paths(
    start=Path.cwd(),
    include_downloading_helpers=True,
    include_anisotropy_toolbox=True,
    include_sc_pos=True,
)

from functions import download_data as download


In [ ]:
# ---------- User configuration ----------
cdf_lib_path = "/Applications/cdf/cdf/lib"   # Set to your local CDF lib path if needed
credentials = None

start_date = "2022-10-01 00:00"
end_date   = "2022-11-01 00:00"  # end is exclusive for daily slicing

save_usual_files = True
save_base = Path(root_dir) / "examples" / "downloaded_intervals" / "SOLO_merged_MAG_daily"

settings = {
    "Data_path": Path(root_dir) / "data",
    "save_destination": Path(root_dir) / "examples" / "downloaded_intervals",
    "sc": "SOLO",
    "in_rtn": 0,
    "use_local_data": False,

    # interval controls
    "start_date": start_date,
    "end_date": end_date,
    "multiple_intervals": False,
    "duration": "24H",
    "Step": "24H",
    "addit_time_around": 1,

    # force 1-minute cadence products
    "part_resol": 60,
    "MAG_resol": 60,
    "upsample_low_freq_ts": False,

    # quality + output
    "overwrite_files": 1,
    "must_have_qtn": False,
    "Max_par_missing": 30,
    "gap_time_threshold": 5,
    "save_all": True,

    # diagnostics needed for beta and sigma_c
    "estimate_derived_param": True,
    "rol_window": "60min",

    # key requirement from user
    "SOLO_use_merged_MAG": True,
    "SOLO_merged_fs": 256,

    "Big_Gaps": {
        "E_big_gaps": 10,
        "SC_pot_big_gaps": 10,
        "Mag_big_gaps": 500,
        "Par_big_gaps": 500,
        "QTN_big_gaps": 10,
    },

    "cut_in_small_windows": {
        "flag": False,
        "njobs": 1,
        "Step": "5s",
        "duration": "30s",
    },
}

vars_2_downnload = {
    "mag": None,
    "swa": None,
    "rpw": None,
    "ephem": None,
}

save_base.mkdir(parents=True, exist_ok=True)
print("Save path:", save_base)


In [ ]:
# Build 1-day, non-overlapping intervals
start_ts = pd.Timestamp(start_date)
end_ts = pd.Timestamp(end_date)

edges = pd.date_range(start=start_ts, end=end_ts, freq="1D")
if len(edges) < 2:
    raise ValueError("Need at least 1 full day in [start_date, end_date].")

intervals = pd.DataFrame({
    "Start": edges[:-1],
    "End": edges[1:]
})

print(f"Generated {len(intervals)} daily intervals (no overlap).")
intervals.head()


In [ ]:
def _folder_name(start_time, end_time):
    tfmt = "%Y-%m-%d_%H-%M-%S"
    return f"{start_time.strftime(tfmt)}_{end_time.strftime(tfmt)}_sc_0"


def _safe_stat(series, reducer="mean"):
    if series is None:
        return np.nan
    series = pd.to_numeric(series, errors="coerce")
    if series.notna().sum() == 0:
        return np.nan
    if reducer == "mean":
        return float(np.nanmean(series.values))
    if reducer == "median":
        return float(np.nanmedian(series.values))
    if reducer == "std":
        return float(np.nanstd(series.values))
    raise ValueError("Unsupported reducer")


rows = []

for ii, row in intervals.iterrows():
    start_time = row["Start"]
    end_time = row["End"]

    (
        big_gaps_SC_pot,
        big_gaps,
        big_gaps_par,
        big_gaps_elec,
        big_gaps_qtn,
        flag_good,
        final_dict,
        general_dict,
        sig_df,
        dfdis,
        misc,
    ) = download.main_function(
        start_time,
        end_time,
        settings,
        vars_2_downnload,
        cdf_lib_path,
        credentials,
    )

    available = int(flag_good == 1 and isinstance(sig_df, pd.DataFrame) and len(sig_df) > 0)

    if available:
        sig_1min = sig_df.resample("1min").mean(numeric_only=True)
        beta = sig_1min.get("beta")
        sigma_c = sig_1min.get("sigma_c")

        beta_mean = _safe_stat(beta, "mean")
        beta_median = _safe_stat(beta, "median")
        sigma_c_abs_mean = _safe_stat(np.abs(sigma_c), "mean")
        sigma_c_abs_median = _safe_stat(np.abs(sigma_c), "median")

        # Save "usual files" for successful intervals
        if save_usual_files:
            folder = save_base / _folder_name(start_time, end_time)
            folder.mkdir(parents=True, exist_ok=True)

            pd.to_pickle(final_dict, folder / "final.pkl")
            pd.to_pickle(general_dict, folder / "general.pkl")
            pd.to_pickle(sig_df, folder / "sig_c_sig_r.pkl")
            pd.to_pickle(big_gaps, folder / "mag_gaps.pkl")
            pd.to_pickle(big_gaps_qtn, folder / "qtn_gaps.pkl")
            pd.to_pickle(big_gaps_par, folder / "par_gaps.pkl")
            pd.to_pickle(big_gaps_SC_pot, folder / "sc_pot_gaps.pkl")
            pd.to_pickle(big_gaps_elec, folder / "elec_gaps.pkl")
            pd.to_pickle(dfdis, folder / "distance.pkl")
            pd.to_pickle(misc, folder / "misc.pkl")

    else:
        beta_mean = np.nan
        beta_median = np.nan
        sigma_c_abs_mean = np.nan
        sigma_c_abs_median = np.nan

    rows.append({
        "Start": start_time,
        "End": end_time,
        "available": available,
        "beta_mean_1min": beta_mean,
        "beta_median_1min": beta_median,
        "abs_sigma_c_mean_1min": sigma_c_abs_mean,
        "abs_sigma_c_median_1min": sigma_c_abs_median,
    })

summary = pd.DataFrame(rows)
summary


In [ ]:
# Persist summary products
summary_path = save_base / "daily_merged_mag_availability_summary.csv"
summary.to_csv(summary_path, index=False)

available_days = int(summary["available"].sum())
total_days = int(len(summary))
print(f"Available days: {available_days}/{total_days}")
print(f"Summary saved to: {summary_path}")

summary.head(20)


## A smarter strategy (recommended before long runs)

If you are scanning long time ranges, do this in two phases:

1. **Pre-screen merged MAG availability only** (cheap):
   - Query merged MAG products day-by-day (no plasma diagnostics yet).
   - Keep only days where merged MAG exists.
2. **Run full diagnostics only on candidate days**:
   - For candidate days, compute 1-minute `beta` and `sigma_c` with the full pipeline.

Why this helps:
- You avoid expensive full-pipeline failures on days with no merged MAG.
- You still get physically consistent `beta` and `sigma_c` from synchronized plasma + MAG data.

If you want, I can add a dedicated pre-screen helper cell that uses SOAR/pySPEDAS metadata first, then auto-runs the full diagnostics only on valid days.
